# K-Means Clustering for Tourist Attractions

Enhanced notebook with:
- Function to test with any K value and visualize on Goa map
- Function to take coordinates list and K as input and display clusters
- Interactive map visualization using Folium

In [ ]:
# Import required libraries
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster
import pandas as pd

print('Libraries loaded!')

## Sample Coordinates (Goa Tourist Attractions)

In [ ]:
# Example coordinates (latitude, longitude) - REPLACE WITH YOUR DATA
coordinates = np.array([
    [15.4922519, 73.7737462],   # Fort Aguada
    [15.3144375, 74.3143073],   # Dudhsagar Falls
    [15.3544232, 73.8838637],   # Velsao Beach
    [15.0887849, 73.9215933],   # Cabo de Rama Fort
    [15.5507728, 74.0264703],   # Harvalem Waterfalls
    [15.6046375, 73.7369631],   # Chapora Fort
    [15.4964204, 73.8091839],   # Reis Magos Fort
    [15.5008429, 73.9114097],   # Basilica of Bom Jesus
    [15.3823119, 73.9288010],   # Kesarval Spring
    [15.5035996, 73.9122204],   # Se Cathedral
])

# Location names for better visualization
location_names = [
    'Fort Aguada',
    'Dudhsagar Falls',
    'Velsao Beach',
    'Cabo de Rama Fort',
    'Harvalem Waterfalls',
    'Chapora Fort',
    'Reis Magos Fort',
    'Basilica of Bom Jesus',
    'Kesarval Spring',
    'Se Cathedral'
]

print(f'Number of locations: {len(coordinates)}')
print(f'\nCoordinates shape: {coordinates.shape}')

## Function 1: Test with Any K and Visualize on Goa Map

This function allows you to test different K values and see the clusters on an interactive map.

In [ ]:
def visualize_clusters_on_map(coords, k, location_names=None, map_title="K-Means Clustering"):
    """
    Perform K-Means clustering and visualize results on an interactive Goa map.
    
    Args:
        coords: numpy array of shape (n, 2) with [latitude, longitude]
        k: number of clusters (days)
        location_names: optional list of location names
        map_title: title for the map
    
    Returns:
        labels: array of cluster labels for each coordinate
        centers: cluster centroids
        map_obj: folium map object
    """
    # Perform K-Means clustering
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(coords)
    centers = kmeans.cluster_centers_
    
    # Calculate map center (mean of all coordinates)
    map_center = [coords[:, 0].mean(), coords[:, 1].mean()]
    
    # Create a Folium map centered on Goa
    m = folium.Map(location=map_center, zoom_start=10, tiles='OpenStreetMap')
    
    # Define colors for clusters
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 
              'darkpurple', 'pink', 'lightblue', 'lightgreen', 'gray', 
              'black', 'lightgray']
    
    # Add markers for each location
    for i, (lat, lon) in enumerate(coords):
        cluster_id = labels[i]
        color = colors[cluster_id % len(colors)]
        
        # Create popup text
        if location_names and i < len(location_names):
            popup_text = f"<b>{location_names[i]}</b><br>Cluster: {cluster_id}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}"
        else:
            popup_text = f"Location {i}<br>Cluster: {cluster_id}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}"
        
        # Add marker
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(popup_text, max_width=200),
            icon=folium.Icon(color=color, icon='info-sign'),
            tooltip=f"Cluster {cluster_id}"
        ).add_to(m)
    
    # Add cluster centroids
    for i, (lat, lon) in enumerate(centers):
        color = colors[i % len(colors)]
        folium.Marker(
            location=[lat, lon],
            popup=f"Centroid {i}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
            icon=folium.Icon(color=color, icon='star'),
            tooltip=f"Centroid {i}"
        ).add_to(m)
    
    # Add title
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); 
                z-index: 1000; background-color: white; padding: 10px; 
                border: 2px solid grey; border-radius: 5px;">
        <h3 style="margin: 0;">{map_title} (K={k})</h3>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Print cluster information
    print(f"\n{'='*60}")
    print(f"K-Means Clustering Results (K={k})")
    print(f"{'='*60}")
    
    for i in range(k):
        cluster_indices = np.where(labels == i)[0]
        print(f"\n--- Cluster {i} (Day {i+1}) ---")
        print(f"Number of locations: {len(cluster_indices)}")
        print(f"Centroid: ({centers[i][0]:.6f}, {centers[i][1]:.6f})")
        print("Locations:")
        for idx in cluster_indices:
            if location_names and idx < len(location_names):
                print(f"  - {location_names[idx]}: ({coords[idx][0]:.6f}, {coords[idx][1]:.6f})")
            else:
                print(f"  - Location {idx}: ({coords[idx][0]:.6f}, {coords[idx][1]:.6f})")
    
    return labels, centers, m

### Test with Different K Values

In [ ]:
# Test with K=3
labels_k3, centers_k3, map_k3 = visualize_clusters_on_map(
    coordinates, k=3, location_names=location_names, map_title="Goa Tourist Attractions"
)

# Display the map
map_k3

In [ ]:
# Test with K=4
labels_k4, centers_k4, map_k4 = visualize_clusters_on_map(
    coordinates, k=4, location_names=location_names, map_title="Goa Tourist Attractions"
)

# Display the map
map_k4

In [ ]:
# Test with K=5
labels_k5, centers_k5, map_k5 = visualize_clusters_on_map(
    coordinates, k=5, location_names=location_names, map_title="Goa Tourist Attractions"
)

# Display the map
map_k5

## Function 2: Cluster Any Coordinates List

This function takes any list of coordinates and K as input and displays the clusters.

In [ ]:
def cluster_and_display(coords_list, k, names=None, save_html=False, filename="clusters_map.html"):
    """
    Cluster a list of coordinates and display results on map and console.
    
    Args:
        coords_list: list of [lat, lon] coordinates or numpy array
        k: number of clusters
        names: optional list of location names
        save_html: if True, save map to HTML file
        filename: name of HTML file to save
    
    Returns:
        dict: Dictionary containing labels, centers, and map object
    """
    # Convert to numpy array if needed
    if not isinstance(coords_list, np.ndarray):
        coords = np.array(coords_list)
    else:
        coords = coords_list
    
    # Validate input
    if coords.shape[1] != 2:
        raise ValueError("Coordinates must have shape (n, 2) with [latitude, longitude]")
    
    if k < 1:
        raise ValueError("K must be at least 1")
    
    if k > len(coords):
        raise ValueError(f"K ({k}) cannot be greater than number of coordinates ({len(coords)})")
    
    # Perform clustering
    labels, centers, map_obj = visualize_clusters_on_map(
        coords, k, location_names=names, map_title="Custom Coordinates Clustering"
    )
    
    # Save to HTML if requested
    if save_html:
        map_obj.save(filename)
        print(f"\nMap saved to: {filename}")
    
    return {
        'labels': labels,
        'centers': centers,
        'map': map_obj,
        'coordinates': coords
    }

### Example: Using Custom Coordinates

In [ ]:
# Example 1: Custom coordinates list
custom_coords = [
    [15.4922519, 73.7737462],   # Fort Aguada
    [15.3144375, 74.3143073],   # Dudhsagar Falls
    [15.3544232, 73.8838637],   # Velsao Beach
    [15.0887849, 73.9215933],   # Cabo de Rama Fort
    [15.5507728, 74.0264703],   # Harvalem Waterfalls
]

custom_names = [
    'Fort Aguada',
    'Dudhsagar Falls',
    'Velsao Beach',
    'Cabo de Rama Fort',
    'Harvalem Waterfalls'
]

# Cluster into 2 groups
result = cluster_and_display(custom_coords, k=2, names=custom_names)

# Display the map
result['map']

In [ ]:
# Example 2: Save map to HTML file
result_saved = cluster_and_display(
    custom_coords, 
    k=2, 
    names=custom_names,
    save_html=True,
    filename="custom_clusters_map.html"
)

## Additional: Elbow Method to Find Optimal K

In [ ]:
def find_optimal_k(coords, max_k=10):
    """
    Use the Elbow Method to find optimal number of clusters.
    
    Args:
        coords: numpy array of shape (n, 2) with [latitude, longitude]
        max_k: maximum K to test
    
    Returns:
        inertias: list of inertia values for each K
    """
    inertias = []
    K_range = range(1, min(max_k + 1, len(coords) + 1))
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(coords)
        inertias.append(kmeans.inertia_)
    
    # Plot the elbow curve
    plt.figure(figsize=(10, 6))
    plt.plot(K_range, inertias, 'bo-')
    plt.xlabel('Number of Clusters (K)')
    plt.ylabel('Inertia (Within-cluster sum of squares)')
    plt.title('Elbow Method for Optimal K')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return list(inertias)

# Find optimal K for our coordinates
inertias = find_optimal_k(coordinates, max_k=8)

## Visualization: Compare Different K Values

In [ ]:
def compare_k_values(coords, k_values, location_names=None):
    """
    Compare clustering results for different K values side by side.
    
    Args:
        coords: numpy array of shape (n, 2) with [latitude, longitude]
        k_values: list of K values to compare
        location_names: optional list of location names
    """
    n_plots = len(k_values)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
    
    if n_plots == 1:
        axes = [axes]
    
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray']
    
    for idx, k in enumerate(k_values):
        ax = axes[idx]
        
        # Perform clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(coords)
        centers = kmeans.cluster_centers_
        
        # Plot clusters
        for i in range(k):
            cluster_points = coords[labels == i]
            ax.scatter(cluster_points[:, 1], cluster_points[:, 0], 
                      c=colors[i % len(colors)], label=f'Cluster {i}', 
                      s=100, edgecolors='black', alpha=0.7)
        
        # Plot centroids
        ax.scatter(centers[:, 1], centers[:, 0],
                  c='black', marker='X', s=200, label='Centroids', 
                  edgecolors='white', linewidth=2)
        
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title(f'K={k}')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('K-Means Clustering Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Compare K=2, K=3, K=4
compare_k_values(coordinates, k_values=[2, 3, 4], location_names=location_names)

## Summary Statistics

In [ ]:
def print_cluster_summary(coords, k, location_names=None):
    """
    Print detailed summary of clustering results.
    
    Args:
        coords: numpy array of shape (n, 2) with [latitude, longitude]
        k: number of clusters
        location_names: optional list of location names
    """
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(coords)
    centers = kmeans.cluster_centers_
    
    print(f"\n{'='*70}")
    print(f"CLUSTERING SUMMARY (K={k})")
    print(f"{'='*70}")
    print(f"Total locations: {len(coords)}")
    print(f"Number of clusters: {k}")
    print(f"\nCluster Distribution:")
    print(f"{'-'*70}")
    
    for i in range(k):
        cluster_indices = np.where(labels == i)[0]
        cluster_coords = coords[labels == i]
        
        # Calculate cluster statistics
        lat_range = (cluster_coords[:, 0].min(), cluster_coords[:, 0].max())
        lon_range = (cluster_coords[:, 1].min(), cluster_coords[:, 1].max())
        
        print(f"\nCluster {i}:")
        print(f"  Count: {len(cluster_indices)} locations")
        print(f"  Centroid: ({centers[i][0]:.6f}, {centers[i][1]:.6f})")
        print(f"  Latitude range: {lat_range[0]:.6f} to {lat_range[1]:.6f}")
        print(f"  Longitude range: {lon_range[0]:.6f} to {lon_range[1]:.6f}")
        print(f"  Locations:")
        
        for idx in cluster_indices:
            if location_names and idx < len(location_names):
                print(f"    - {location_names[idx]}")
            else:
                print(f"    - Location {idx}")
    
    print(f"\n{'='*70}")

# Print summary for K=3
print_cluster_summary(coordinates, k=3, location_names=location_names)

## Interactive: Test Any K Value

In [ ]:
# Change this value to test different K values
TEST_K = 3  # <-- Change this to any value you want

# Visualize with the selected K
labels_test, centers_test, map_test = visualize_clusters_on_map(
    coordinates, k=TEST_K, location_names=location_names, map_title="Interactive Test"
)

# Display the map
map_test

## Save Maps to Files

In [ ]:
# Save all generated maps to HTML files
map_k3.save('goa_clusters_k3.html')
map_k4.save('goa_clusters_k4.html')
map_k5.save('goa_clusters_k5.html')

print("Maps saved:")
print("  - goa_clusters_k3.html")
print("  - goa_clusters_k4.html")
print("  - goa_clusters_k5.html")